# 05 — Intent Dataset Evaluation (Bitext)

## 1. Introduction and objectives

CallVerse needs **customer intents**: the reasons customers contact support ("Where is my order?", "I need a refund", …). Olist has none. This notebook (Phase 1E-a) evaluates the **Bitext Retail eCommerce** dataset as the source of intent labels and example customer wording for the future synthetic contacts.

Objectives:

1. Check the structure and quality of the file.
2. Describe the intents, categories and message styles.
3. **Propose** which intents are in scope for CallVerse.
4. **Propose** which Olist order situation could trigger each order-related intent, and count those situations in the Phase 1D order table.

The scope and mapping in sections 7–8 are **proposals** to be reviewed before contact rates are defined. Nothing here generates contacts.

> **What Bitext is and is not.**
> - It is a *hybrid synthetic* dataset generated by Bitext with NLP/NLG tools, not real customer traffic.
> - It provides realistic **wording** and **intent labels**, not **frequencies**: each intent has roughly the same number of examples by design.
> - It is in English, while Olist is a Brazilian marketplace.
> - Licence: **CDLA-Sharing 1.0**. Attribution is required, and shared derivatives must use the same licence.

## 2. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
BITEXT_DIR = PROJECT_ROOT / "data" / "raw" / "bitext"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Bitext directory:", BITEXT_DIR, "exists:", BITEXT_DIR.exists())
if not BITEXT_DIR.exists():
    raise FileNotFoundError(f"Dataset directory missing: {BITEXT_DIR}. Put the Bitext CSV in data/raw/bitext/.")

Project root: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Bitext directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\raw\bitext exists: True


## 3. Load the dataset

The file is read only; nothing in `data/raw/` is modified. The file is downloaded from Hugging Face (`bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset`).

In [2]:
BITEXT_FILE = BITEXT_DIR / "bitext-retail-ecommerce-llm-chatbot-training-dataset.csv"
if not BITEXT_FILE.exists():
    raise FileNotFoundError(
        f"Missing {BITEXT_FILE.name} in {BITEXT_DIR}. Download it from "
        "https://huggingface.co/datasets/bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset"
    )

bitext = pd.read_csv(BITEXT_FILE)
print("File:", BITEXT_FILE.name, "| size (MB):", round(BITEXT_FILE.stat().st_size / 1024**2, 2))
print("Rows:", len(bitext), "| columns:", len(bitext.columns))
display(pd.DataFrame({"column": bitext.columns, "dtype": bitext.dtypes.astype(str).values}))
display(bitext.head(3))

File: bitext-retail-ecommerce-llm-chatbot-training-dataset.csv | size (MB): 40.65
Rows: 44884 | columns: 5


,column,dtype
0,instruction,object
1,intent,object
2,category,object
3,tags,object
4,response,object


,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...


Column meanings (from the dataset card):

| Column | Meaning |
|---|---|
| `instruction` | Example customer message |
| `intent` | Intent label (46 expected) |
| `category` | Group of intents (13 expected) |
| `tags` | Letter codes describing language variation (see section 6.2) |
| `response` | Example assistant reply, with `{{placeholders}}` to fill in |

## 4. Structure and quality checks

We check missing values, empty texts, duplicates, and whether each intent belongs to exactly one category.

In [3]:
quality_rows = []
for column in bitext.columns:
    text = bitext[column].astype(str).str.strip()
    quality_rows.append({
        "column": column,
        "missing": int(bitext[column].isna().sum()),
        "empty_after_strip": int((bitext[column].notna() & (text == "")).sum()),
        "distinct_values": int(bitext[column].nunique()),
    })
display(pd.DataFrame(quality_rows))

print("Exact duplicate rows:", int(bitext.duplicated().sum()))
print("Duplicate instructions:", int(bitext["instruction"].duplicated().sum()))

# Same message labelled with different intents would make the labels ambiguous
normalized_instruction = bitext["instruction"].str.lower().str.strip()
labels_per_message = bitext.groupby(normalized_instruction)["intent"].nunique()
print("Messages (case-insensitive) with more than one intent:", int((labels_per_message > 1).sum()))

categories_per_intent = bitext.groupby("intent")["category"].nunique()
print("Intents assigned to more than one category:", int((categories_per_intent > 1).sum()))

,column,missing,empty_after_strip,distinct_values
0,instruction,0,0,44884
1,intent,0,0,46
2,category,0,0,13
3,tags,0,0,409
4,response,0,0,44867


Exact duplicate rows: 0
Duplicate instructions: 0
Messages (case-insensitive) with more than one intent: 0
Intents assigned to more than one category: 0


## 5. Intents and categories

### 5.1 Examples per category and intent

In [4]:
category_counts = bitext.groupby("category").agg(intents=("intent", "nunique"), examples=("intent", "size"))
display(category_counts.sort_values("examples", ascending=False))

intent_counts = bitext.groupby(["category", "intent"]).size().rename("examples").reset_index()
with pd.option_context("display.max_rows", None):
    display(intent_counts.sort_values(["category", "intent"]))

print("Examples per intent - min:", intent_counts["examples"].min(),
      "| median:", intent_counts["examples"].median(), "| max:", intent_counts["examples"].max())
print("Intents with fewer than 900 examples:")
display(intent_counts[intent_counts["examples"] < 900])

,intents,examples
category,,
RETURNS,7,6925
PRODUCT,7,6679
DELIVERY,7,6595
ACCOUNT,5,4950
ORDER,4,3945
PAYMENT,3,2981
FEEDBACK,3,2980
APP_WEBSITE,2,1993
CONTACT,2,1986


,category,intent,examples
0,ACCOUNT,change_account,987
1,ACCOUNT,close_account,995
2,ACCOUNT,open_account,987
3,ACCOUNT,order_history,988
4,ACCOUNT,recover_password,993
5,APP_WEBSITE,technical_issue,994
6,APP_WEBSITE,use_app,999
7,CART,add_product,957
8,CART,remove_product,993
9,CONTACT,customer_service,992


Examples per intent - min: 721 | median: 992.5 | max: 1000
Intents with fewer than 900 examples:


,category,intent,examples
14,DELIVERY,missing_item,721
29,PRODUCT,availability_in_store,756


The intents are **almost perfectly balanced** by construction. This is ideal for training an intent classifier, but it says nothing about how often each intent happens in a real contact center. The contact mix for CallVerse must therefore come from documented assumptions linked to Olist order situations, not from Bitext counts.

## 6. Message characteristics

### 6.1 Message length

In [5]:
bitext["instruction_words"] = bitext["instruction"].str.split().str.len()
bitext["response_words"] = bitext["response"].str.split().str.len()

display(bitext[["instruction_words", "response_words"]].describe().round(1))
print("Median instruction length (words) by category:")
display(bitext.groupby("category")["instruction_words"].median().sort_values().to_frame("median_words"))

,instruction_words,response_words
count,44884.0,44884.0
mean,11.8,144.1
std,2.9,51.4
min,1.0,9.0
25%,10.0,110.0
50%,12.0,142.0
75%,14.0,174.0
max,24.0,415.0


Median instruction length (words) by category:


,median_words
category,
ACCOUNT,11.0
CONTACT,11.0
FEEDBACK,11.0
PAYMENT,11.0
USER,11.0
APP_WEBSITE,12.0
DELIVERY,12.0
RETURNS,12.0
SALES,12.0


### 6.2 Language-variation tags

Each letter in `tags` marks a variation in the message (dataset card): **B** basic syntax, **C** coordinated (several ideas), **I** question form, **K** keyword-only, **L** semantic variation (synonyms), **M** morphological variation, **N** negation, **P** politeness, **Q** colloquial, **W** offensive language, **E** abbreviations, **Z** errors and typos.

These variations matter for CallVerse: simulated customers should not all write perfect, polite sentences.

In [6]:
TAG_MEANINGS = {
    "B": "basic syntax", "C": "coordinated", "I": "question form", "K": "keyword only",
    "L": "semantic variation", "M": "morphological variation", "N": "negation", "P": "politeness",
    "Q": "colloquial", "W": "offensive language", "E": "abbreviations", "Z": "errors and typos",
}

tag_rows = []
for letter, meaning in TAG_MEANINGS.items():
    has_tag = bitext["tags"].str.contains(letter, regex=False)
    tag_rows.append({"tag": letter, "meaning": meaning, "messages": int(has_tag.sum()),
                     "percent": round(has_tag.mean() * 100, 1)})
display(pd.DataFrame(tag_rows).sort_values("messages", ascending=False))

unknown_letters = set("".join(bitext["tags"].unique())) - set(TAG_MEANINGS)
print("Tag letters not documented above:", sorted(unknown_letters))

,tag,meaning,messages,percent
0,B,basic syntax,44884,100.0
1,C,coordinated,35872,79.9
2,I,question form,31863,71.0
4,L,semantic variation,31275,69.7
11,Z,errors and typos,22940,51.1
9,W,offensive language,22311,49.7
8,Q,colloquial,22281,49.6
7,P,politeness,19896,44.3
5,M,morphological variation,8286,18.5
10,E,abbreviations,2835,6.3


Tag letters not documented above: []


In [7]:
# A few messages with offensive language (W) and with typos (Z)
print("Offensive-language examples (W):")
display(bitext.loc[bitext["tags"].str.contains("W"), ["instruction", "intent"]].sample(3, random_state=7))
print("Typo examples (Z):")
display(bitext.loc[bitext["tags"].str.contains("Z"), ["instruction", "intent"]].sample(3, random_state=7))

Offensive-language examples (W):


,instruction,intent
17280,id liek to make a fucking transfer can uhelp me,pay
28786,I'd like to complete my fucking personal infor...,request_right_to_rectification
13143,need to exchange a fucking product i purchased...,exchange_product_in_store


Typo examples (Z):


,instruction,intent
21884,"my product doesn't work corretly, how could I ...",product_issue
14182,"I would like to speak with a fucking human, co...",human_agent
33875,I'm looking for information about the fucking ...,sales_period


### 6.3 Placeholders

Bitext replaces specific values with `{{placeholders}}`, for example `{{Order Number}}`. The counts below show where they occur: if they appear only in the example replies, the customer messages can be used as they are, while replies would need the placeholders filled with Olist values (or removed). It also means a customer message never names a specific order, so linking a simulated contact to an Olist order is up to the simulation.

In [8]:
placeholder_pattern = r"\{\{([^}]+)\}\}"

instruction_placeholders = bitext["instruction"].str.findall(placeholder_pattern).explode().dropna()
response_placeholders = bitext["response"].str.findall(placeholder_pattern).explode().dropna()

print("Instructions containing a placeholder:", int(bitext["instruction"].str.contains(r"\{\{", regex=True).sum()))
print("Responses containing a placeholder:", int(bitext["response"].str.contains(r"\{\{", regex=True).sum()))
print("Distinct placeholders in instructions:", instruction_placeholders.nunique())
print("Distinct placeholders in responses:", response_placeholders.nunique())

print("Most common placeholders in instructions:")
display(instruction_placeholders.value_counts().head(10).to_frame("occurrences"))
print("Most common placeholders in responses:")
display(response_placeholders.value_counts().head(10).to_frame("occurrences"))

Instructions containing a placeholder: 0
Responses containing a placeholder: 32579
Distinct placeholders in instructions: 0
Distinct placeholders in responses: 2969
Most common placeholders in instructions:


,occurrences
instruction,


Most common placeholders in responses:


,occurrences
response,
Company Website URL,12715
Customer Support Phone Number,11789
Order Number,9084
Customer Support Working Hours,5259
Order History,4591
Customer Support Team,3246
Email Address,2046
My Orders,1999
Customer Support,1184


### 6.4 One example per intent

In [9]:
examples_per_intent = bitext.groupby("intent").sample(1, random_state=42)[["category", "intent", "instruction"]]
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(examples_per_intent.sort_values(["category", "intent"]))

,category,intent,instruction
5036,ACCOUNT,change_account,I'd like to make a change to my fucking profile where to do it
6754,ACCOUNT,close_account,i got to delete my fucking user account were do i do it
15532,ACCOUNT,open_account,io got to open a fucking user account how to do it
16691,ACCOUNT,order_history,"I'd like to see the fucking purchase history, can I get some help?"
22725,ACCOUNT,recover_password,"my profile has been hacked , how could I reset my password?"
40187,APP_WEBSITE,technical_issue,"the mbile app doesn't work, help me reporting the fucking issue"
43770,APP_WEBSITE,use_app,id like to use ur fucking mobile app where do i do it
835,CART,add_product,"I would like to add a fucking item to the cart, how to do it?"
25393,CART,remove_product,"I have to remove a fucking item ffom the cart, how to do it?"
8501,CONTACT,customer_service,I'd like to call customer support how can i do it


## 7. Proposed scope for CallVerse

Olist is an **online** marketplace that delivers orders to customers. We propose three scope levels for each intent:

| Scope | Meaning |
|---|---|
| `order_linked` | The contact concerns a specific order, and an Olist order situation could plausibly trigger it. These intents can be driven by Olist data. |
| `general` | A plausible contact for an online shop, but not tied to a specific order situation in Olist (account, website, policies, products). Its volume would need a base-rate assumption. |
| `out_of_scope` | Refers to physical stores, which an online marketplace does not have. |

This is a **proposal**, not a final decision.

In [10]:
# intent -> (proposed scope, reason)
PROPOSED_SCOPE = {
    # ORDER
    "track_order": ("order_linked", "Customer asks where the order is"),
    "cancel_order": ("order_linked", "Olist has canceled orders"),
    "change_order": ("order_linked", "Change before shipping; any order in progress"),
    "request_invoice": ("order_linked", "Concerns a specific order"),
    # DELIVERY
    "track_delivery": ("order_linked", "Same need as track_order, delivery wording"),
    "delivery_time": ("order_linked", "Promised versus actual delivery dates"),
    "delivery_issue": ("order_linked", "Late or overdue delivery"),
    "damaged_delivery": ("order_linked", "Post-delivery problem; no direct Olist signal"),
    "missing_item": ("order_linked", "Post-delivery problem; no direct Olist signal"),
    "wrong_item": ("order_linked", "Post-delivery problem; no direct Olist signal"),
    "shipping_costs": ("general", "Policy question; freight exists but no problem signal"),
    # RETURNS
    "request_refund": ("order_linked", "Canceled/unavailable orders, problems"),
    "refund_status": ("order_linked", "Follow-up after a refund request"),
    "return_product": ("order_linked", "Post-delivery dissatisfaction"),
    "return_product_online": ("order_linked", "Post-delivery dissatisfaction"),
    "refund_policy": ("general", "Policy question"),
    "return_policy": ("general", "Policy question"),
    "return_product_in_store": ("out_of_scope", "Physical store"),
    # PRODUCT
    "exchange_product": ("order_linked", "Post-delivery dissatisfaction"),
    "product_issue": ("order_linked", "Post-delivery problem with the product"),
    "product_information": ("general", "Pre-purchase question"),
    "availability": ("general", "Pre-purchase question"),
    "availability_online": ("general", "Pre-purchase question"),
    "availability_in_store": ("out_of_scope", "Physical store"),
    "exchange_product_in_store": ("out_of_scope", "Physical store"),
    # PAYMENT
    "payment_issue": ("order_linked", "Unusual payments exist in Olist"),
    "pay": ("general", "How to pay; not a problem signal"),
    "payment_methods": ("general", "Policy question"),
    # ACCOUNT
    "order_history": ("general", "Account question"),
    "change_account": ("general", "Account question"),
    "close_account": ("general", "Account question"),
    "open_account": ("general", "Account question"),
    "recover_password": ("general", "Account question"),
    # Other categories
    "add_product": ("general", "Shopping-cart help"),
    "remove_product": ("general", "Shopping-cart help"),
    "customer_service": ("general", "Asks how to reach support"),
    "human_agent": ("general", "Asks for a human; relevant to escalation"),
    "technical_issue": ("general", "Website or app problem"),
    "use_app": ("general", "Website or app help"),
    "submit_feedback": ("general", "Feedback; Olist reviews are a separate channel"),
    "submit_product_feedback": ("general", "Feedback; Olist reviews are a separate channel"),
    "submit_product_idea": ("general", "Feedback"),
    "sales_period": ("general", "Promotions question; linked to the future promo calendar"),
    "store_location": ("out_of_scope", "Physical store"),
    "store_opening_hours": ("out_of_scope", "Physical store"),
    "request_right_to_rectification": ("general", "Data-protection request"),
}

dataset_intents = set(bitext["intent"].unique())
missing_in_proposal = sorted(dataset_intents - set(PROPOSED_SCOPE))
unknown_in_proposal = sorted(set(PROPOSED_SCOPE) - dataset_intents)
if missing_in_proposal or unknown_in_proposal:
    raise ValueError(f"Scope table out of sync. Missing: {missing_in_proposal}; unknown: {unknown_in_proposal}")

scope_by_intent = {}
reason_by_intent = {}
for intent, (scope, reason) in PROPOSED_SCOPE.items():
    scope_by_intent[intent] = scope
    reason_by_intent[intent] = reason

intent_catalog = intent_counts.copy()
intent_catalog["proposed_scope"] = intent_catalog["intent"].map(scope_by_intent)
intent_catalog["scope_reason"] = intent_catalog["intent"].map(reason_by_intent)

scope_summary = intent_catalog.groupby("proposed_scope").agg(intents=("intent", "size"), examples=("examples", "sum"))
display(scope_summary)

,intents,examples
proposed_scope,,
general,24,23742
order_linked,17,16485
out_of_scope,5,4657


## 8. Proposed link between order-linked intents and Olist situations

For each `order_linked` intent we propose an Olist **signal**: a column of the Phase 1D order table that identifies orders in a situation where this contact is plausible. We then count those orders inside the analysis window.

Signal strength:

- **direct**: the Olist data record the situation itself (for example, an overdue order).
- **weak**: only an indirect hint (for example, a low review with a comment suggests a product problem but does not say which).
- **base_rate**: no specific signal; any order could produce the contact, so only a base rate can be assumed.

**Important:**
- A signal identifies orders *at risk* of a contact. It does not mean a contact happened. How many of these orders lead to a contact will be an explicit assumption in Phase 1E-c.
- Some signals use hindsight. A review is written after delivery, and the final status is known only at the end. For simulation they define *which orders are affected*, while the contact time must be set from the order timeline.

In [11]:
ANALYTICAL_FILE = PROCESSED_DIR / "olist_orders_analytical.parquet"
if not ANALYTICAL_FILE.exists():
    raise FileNotFoundError(f"Missing {ANALYTICAL_FILE.name}. Run notebooks 03 and 04 first.")

orders = pd.read_parquet(ANALYTICAL_FILE)
window_orders = orders[orders["in_analysis_window"]].copy()
print("Orders in the analysis window:", len(window_orders))

# Olist situations (boolean masks on the window orders)
# fillna(False): an unknown value (<NA>) does not count as being in the situation
late_delivery = window_orders["delivered_after_estimated_date"].fillna(False)
overdue = window_orders["overdue_without_delivery"]
late_or_overdue = late_delivery | overdue
seller_late = window_orders["carrier_after_shipping_limit"].fillna(False)
canceled_or_unavailable = window_orders["is_canceled"] | window_orders["is_unavailable"]
low_review_with_comment = (window_orders["review_is_low_score"].fillna(False)
                           & window_orders["review_has_message"].fillna(False))
delivered_low_review = window_orders["has_customer_delivery_timestamp"] & low_review_with_comment
payment_anomaly = ((window_orders["payment_minus_order_value"].abs() > 1)
                   | (window_orders["primary_payment_type"] == "not_defined"))
in_progress_before_handoff = ~window_orders["has_carrier_handoff_timestamp"] & ~canceled_or_unavailable
all_orders = pd.Series(True, index=window_orders.index)

OLIST_SIGNALS = {
    "late_or_overdue": ("Delivered after the estimated date, or overdue without delivery", late_or_overdue),
    "late_overdue_or_seller_late": ("Late/overdue, or the seller handed over after the deadline",
                                    late_or_overdue | seller_late),
    "canceled_or_unavailable": ("Final status canceled or unavailable", canceled_or_unavailable),
    "delivered_low_review_with_comment": ("Delivered, latest review 1-2 stars with a comment", delivered_low_review),
    "payment_anomaly": ("Payment differs from order value by > 1 BRL, or type not_defined", payment_anomaly),
    "not_yet_shipped": ("Not handed to the carrier (and not canceled/unavailable)", in_progress_before_handoff),
    "any_order": ("Any order in the window", all_orders),
}

signal_rows = []
for signal_name, (description, mask) in OLIST_SIGNALS.items():
    signal_rows.append({"signal": signal_name, "description": description,
                        "orders_in_window": int(mask.sum()),
                        "percent_of_window": round(mask.mean() * 100, 2)})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(signal_rows))

Orders in the analysis window: 99092


,signal,description,orders_in_window,percent_of_window
0,late_or_overdue,"Delivered after the estimated date, or overdue without delivery",8228,8.30
1,late_overdue_or_seller_late,"Late/overdue, or the seller handed over after the deadline",15007,15.14
2,canceled_or_unavailable,Final status canceled or unavailable,1182,1.19
3,delivered_low_review_with_comment,"Delivered, latest review 1-2 stars with a comment",9194,9.28
4,payment_anomaly,"Payment differs from order value by > 1 BRL, or type not_defined",246,0.25
5,not_yet_shipped,Not handed to the carrier (and not canceled/unavailable),604,0.61
6,any_order,Any order in the window,99092,100.00


In [12]:
# intent -> (Olist signal, strength)
PROPOSED_SIGNAL = {
    "track_order": ("late_or_overdue", "direct"),
    "track_delivery": ("late_or_overdue", "direct"),
    "delivery_time": ("late_or_overdue", "direct"),
    "delivery_issue": ("late_overdue_or_seller_late", "direct"),
    "cancel_order": ("canceled_or_unavailable", "direct"),
    "request_refund": ("canceled_or_unavailable", "direct"),
    "refund_status": ("canceled_or_unavailable", "direct"),
    "damaged_delivery": ("delivered_low_review_with_comment", "weak"),
    "missing_item": ("delivered_low_review_with_comment", "weak"),
    "wrong_item": ("delivered_low_review_with_comment", "weak"),
    "product_issue": ("delivered_low_review_with_comment", "weak"),
    "return_product": ("delivered_low_review_with_comment", "weak"),
    "return_product_online": ("delivered_low_review_with_comment", "weak"),
    "exchange_product": ("delivered_low_review_with_comment", "weak"),
    "payment_issue": ("payment_anomaly", "weak"),
    "change_order": ("not_yet_shipped", "base_rate"),
    "request_invoice": ("any_order", "base_rate"),
}

order_linked_intents = set(intent_catalog.loc[intent_catalog["proposed_scope"] == "order_linked", "intent"])
if order_linked_intents != set(PROPOSED_SIGNAL):
    raise ValueError("Every order_linked intent needs exactly one proposed signal.")

signal_by_intent = {}
strength_by_intent = {}
for intent, (signal_name, strength) in PROPOSED_SIGNAL.items():
    signal_by_intent[intent] = signal_name
    strength_by_intent[intent] = strength

signal_counts = {}
for signal_name, (description, mask) in OLIST_SIGNALS.items():
    signal_counts[signal_name] = int(mask.sum())

# Intents without a proposed signal (general, out_of_scope) stay empty
intent_catalog["olist_signal"] = intent_catalog["intent"].map(signal_by_intent)
intent_catalog["signal_strength"] = intent_catalog["intent"].map(strength_by_intent)
intent_catalog["signal_orders_in_window"] = intent_catalog["olist_signal"].map(signal_counts).astype("Int64")

linked = intent_catalog[intent_catalog["proposed_scope"] == "order_linked"]
with pd.option_context("display.max_rows", None):
    display(linked[["category", "intent", "olist_signal", "signal_strength", "signal_orders_in_window"]]
            .sort_values(["signal_strength", "olist_signal", "intent"]))
display(linked.groupby("signal_strength").size().rename("intents").to_frame())

,category,intent,olist_signal,signal_strength,signal_orders_in_window
23,ORDER,request_invoice,any_order,base_rate,99092
22,ORDER,change_order,not_yet_shipped,base_rate,604
21,ORDER,cancel_order,canceled_or_unavailable,direct,1182
36,RETURNS,refund_status,canceled_or_unavailable,direct,1182
37,RETURNS,request_refund,canceled_or_unavailable,direct,1182
13,DELIVERY,delivery_time,late_or_overdue,direct,8228
16,DELIVERY,track_delivery,late_or_overdue,direct,8228
24,ORDER,track_order,late_or_overdue,direct,8228
12,DELIVERY,delivery_issue,late_overdue_or_seller_late,direct,15007
11,DELIVERY,damaged_delivery,delivered_low_review_with_comment,weak,9194


,intents
signal_strength,
base_rate,2
direct,7
weak,8


**Reading the table.**

- **Direct signals** cover the clearest contact drivers in the CallVerse architecture: "Where is my order?" (late or overdue) and "I need a refund" (canceled or unavailable).
- **Weak signals** concern post-delivery problems (wrong, damaged or missing items). Olist does not record them, so a low review with a comment is the best available hint.
- **Base-rate intents** can happen on any order and need a simple rate assumption.

Several intents share one signal, for example `track_order`, `track_delivery` and `delivery_time`. Phase 1E-c will need a rule for splitting contacts between them.

## 9. Limitations

- **Synthetic wording.** Bitext messages are generated, not observed. They provide realistic variety but no real-world frequencies, timing, or customer history.
- **Balanced intents.** About 1,000 examples per intent; the CallVerse contact mix must come from documented assumptions.
- **Language mismatch.** Bitext is English; Olist customers are Brazilian. Either CallVerse is presented as an English-language contact center, or messages need translation. This is a design choice to document.
- **Placeholders.** Example replies contain `{{placeholders}}` (section 6.3) that must be filled with Olist values or removed before use.
- **Offensive language.** A large share of messages contain swearing by design (`W` tag, section 6.2). This is useful for realism, but CallVerse may need to filter or down-weight these messages in demos, and the agents must handle them.
- **Tags not observed.** The card lists keyword mode (`K`) and negation (`N`) among the tags used, but section 6.2 shows whether they actually occur in the file.
- **Licence.** CDLA-Sharing 1.0: credit Bitext, and share derived data under the same licence.
- **Mapping uncertainty.** Weak signals and base-rate intents rest on assumptions, not on Olist evidence.

## 10. Export the intent catalog

The intent catalog (one row per Bitext intent with example count, proposed scope, proposed Olist signal and signal strength) is saved for Phase 1E-b/c. The Bitext messages themselves are not copied; the raw file remains the source.

In [13]:
catalog_columns = ["category", "intent", "examples", "proposed_scope", "scope_reason",
                   "olist_signal", "signal_strength", "signal_orders_in_window"]
catalog_path = PROCESSED_DIR / "bitext_intent_catalog.csv"
intent_catalog[catalog_columns].sort_values(["proposed_scope", "category", "intent"]).to_csv(catalog_path, index=False)

reloaded = pd.read_csv(catalog_path)
print("Saved:", catalog_path.relative_to(PROJECT_ROOT), "| rows:", len(reloaded), "| columns:", len(reloaded.columns))
display(reloaded.groupby(["proposed_scope", "signal_strength"], dropna=False).size().rename("intents").to_frame())

Saved: data\processed\bitext_intent_catalog.csv | rows: 46 | columns: 8


intents
proposed_scope signal_strength         
general        NaN                   24
order_linked   base_rate              2
               direct                 7
               weak                   8
out_of_scope   NaN                    5

## 11. Summary and decisions needed

**Findings**

- The Bitext Retail eCommerce file is complete and clean: no missing values, no duplicates, and each intent belongs to one category.
- Its 46 intents cover the contact reasons in the CallVerse architecture ("Where is my order?", "I need a refund", "Wrong item received").
- Intents are balanced by design, so Bitext provides wording and labels but not contact frequencies.
- Messages vary in style (questions, colloquial language, typos, offensive language), which is useful for realistic simulated customers.

**Proposals to review** (sections 7–8, saved in `bitext_intent_catalog.csv`)

- Scope: `order_linked`, `general` or `out_of_scope` for every intent; physical-store intents are excluded.
- One Olist signal and signal strength for every order-linked intent.

**Decisions needed before Phase 1E-b/c**

1. Accept or adjust the proposed scope and signals.
2. Decide whether `general` intents are simulated (with a base rate) or left out at first.
3. Decide the language of the simulated contact center (English with Bitext as-is, or translated).
4. Then define contact rates: the share of orders in each situation that contact support, and when.

> Bitext messages are themselves synthetic. Any CallVerse contact built from them is synthetic and must be labelled as such, never presented as observed Olist or Bitext traffic.